In [1]:
import sys
sys.path.append('..')

In [2]:
from pyboy import PyBoy
from base64 import b64encode
import io
import requests

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama


In [3]:
MODEL_ENDPOINT = 'http://127.0.0.1:8080/v1'

In [4]:
from langchain_openai import ChatOpenAI

class ChatLlamaCpp(ChatOpenAI):
    def _create_chat_result(self, response, generation_info=None):
        result = super()._create_chat_result(response, generation_info)
        choices = (response if isinstance(response, dict)
                   else response.model_dump()).get("choices", [])
        for gen, choice in zip(result.generations, choices):
            if rc := choice.get("message", {}).get("reasoning_content"):
                gen.message.additional_kwargs["reasoning_content"] = rc
        return result


In [5]:
pyboy = PyBoy('./roms/pkmsil.gbc', window='null')

In [6]:
with open('./states/bedroom.state', 'rb') as f:
    pyboy.load_state(f)

In [7]:
def load_state(instance: PyBoy, state_file_loc: str):
    with open(state_file_loc, 'rb') as f:
        instance.load_state(f)    

In [8]:
load_state(pyboy, './states/bedroom.state')

In [9]:
def base_64_encode_image(image):
    f = io.BytesIO()
    pyboy.screen.image.save(f, format='png')
    encoded = b64encode(f.getvalue()).decode('utf8')
    return encoded

In [10]:
# resp = requests.post(MODEL_ENDPOINT, json={'prompt': 'hello world'})

In [16]:
model = ChatLlamaCpp(model='Qwen', base_url=MODEL_ENDPOINT, api_key='nada', reasoning_effort="low")

In [17]:
agent = create_agent( model, system_prompt='you are playing pokemon silver')

In [18]:
msg = HumanMessage(content_blocks=[
    {"type": "text", "text": "What does this screenshot from the game show?"},
    {"type": "image", "base64": base_64_encode_image(pyboy.screen.image), "mime_type": "image/png"},
])

In [19]:
def invoke_agent(agent, prompt: str, img = None):

    message_contents = [
        {"type": "text", "text": prompt}
    ]
    
    if img is not None:
        message_contents.append({"type": "image", "base64": base_64_encode_image(img), "mime_type": "image/png"})


    msg = HumanMessage(content_blocks=message_contents)
    
    return agent.invoke({"messages": [msg]})



In [20]:
invoke_agent(agent, "What does this screenshot from the game show?", pyboy.screen.image)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'What does this screenshot from the game show?'}, {'type': 'image', 'base64': 'iVBORw0KGgoAAAANSUhEUgAAAKAAAACQCAYAAACPtWCAAAAIVUlEQVR4nO2dLZfyOhDH57nnCuR+BCSyEolErkQikSv7EZBIZCWysnIlsnLlyitX4p4rYNgQkr5NYJrw/52zp7tNum3Dn8lk8vZnPp//JQCU+Ef7AcBrAwECVSBAoAoECFSBAIEqECBQBQIEqkCAQBUIEKgCAQJVIECgCgQIVIEAgSoQIFAFAgSq/Cv9Bx/vRF/f3860qp4ivSVdivb9pYgFCORUh/2g65arzfn6BiHtSiIifaH5CCrA2XTq/bYhvT1dwn773iv/Ji8f8hx9gQVMiPKw65TvffVxdy7LMqrruvdRStBGSNu3G+mPsX4ms+mUZtN+Va6W+OgRreC2l0f6+PyxIeLLsizIvVEFj4B8O6wR4sK2siz4NuvLFq3vUUpwAWpXYzGmD7UmbSIwrW1TA8h1/y7PVBRFp+dsInoL+PO2pv/I/610hSF+C1f+LW4Lc3QNg/S1KG0CcVX1XVvhoarXLkQvQKau687xtOVqE7yQF8tPqo/d8//8LG7+DiVAqY/5TPFRanFAKZL7f1YL5/k+rNdrKorCeaRLlcfnfAwpH1P8oXy7riRjATWpDnvaFrNe19THeyvIrJZzKorieuyKK77XhSE9MYfqOD4fUNvBP1Q96sAH3F+KT3Cr5ZzoYgF9ecbSs9GX4BZQuxpu+1ZydRby/twnGwr+f67/6zyX/T6P+XyuwQh2ujbRD8cqiuLOb6kO+1ZnOqSv8/b2GeT/+IS8XG2Ci3wsJBEHrA77mw+oy4dlXyO5P/tyi2U3IboaLKFan0OeX1PcQQToa/rPpo9P3w10fdhfDBXH

In [ ]:
resp = model.invoke("How many rs in strawberry?")
print(resp.additional_kwargs)
print(resp.response_metadata)
print(resp.content_blocks)